# Flow Over a Parameterized Block Example Guide

## Level 1 2D flow
This guide explains how to implement a 2D flow over a parameterized block (chip) using PhysicsNeMo. The example demonstrates how to set up a Navier-Stokes solver for a channel with a rectangular obstacle (chip) and how to apply various boundary and interior constraints, as well as validation against reference data.
<center><img src="images/chip_2d.png" alt="Drawing" style="width:500px" /></center>

## Problem Description
- **Domain:** 2D channel with a rectangular chip (block) inside
- **Physics:** Steady-state incompressible Navier-Stokes equations
- **Goal:** Predict the velocity and pressure fields around the chip

Equation to solve: Navier Stokes (continuity, momentum-x, momentum-y)
## Mathematical Formulation

$$\begin{align*} \text{continuity: }\quad\frac{\partial u}{\partial x} + \frac{\partial v}{\partial y}  = 0\\
\text{momentum-x: }\quad u\frac{\partial u}{\partial x} + v\frac{\partial u}{\partial y} +\frac{1}{\rho}\frac{\partial p}{\partial x} - \nu \frac{\partial^2 u}{\partial x^2} - \nu \frac{\partial^2 u}{\partial y^2} = 0 \\
\text{momentum-y: } \quad u\frac{\partial v}{\partial x} + v\frac{\partial v}{\partial y} +\frac{1}{\rho}\frac{\partial p}{\partial y} - \nu \frac{\partial^2 v}{\partial x^2} - \nu \frac{\partial^2 v}{\partial y^2} = 0 \\
\end{align*}
$$

where:
- Boundary conditions: Parabolic inlet velocity, zero outlet pressure, no slip walls
- Fluid density: $\rho = 1.0$, fluid viscosity: $\nu = 0.02$


## Implementation Steps

### 1. Import Required Libraries
```python
import os
import warnings

import numpy as np
from sympy import Symbol, Eq, And, Or

import physicsnemo.sym
from physicsnemo.sym.hydra import to_absolute_path, instantiate_arch, PhysicsNeMoConfig
from physicsnemo.sym.utils.io import csv_to_dict
from physicsnemo.sym.solver import Solver
from physicsnemo.sym.domain import Domain
from physicsnemo.sym.geometry.primitives_2d import Rectangle, Line, Channel2D
from physicsnemo.sym.utils.sympy.functions import parabola
from physicsnemo.sym.eq.pdes.navier_stokes import NavierStokes
from physicsnemo.sym.eq.pdes.basic import NormalDotVec
from physicsnemo.sym.domain.constraint import (
    PointwiseBoundaryConstraint,
    PointwiseInteriorConstraint,
    IntegralBoundaryConstraint,
)

from physicsnemo.sym.domain.validator import PointwiseValidator
from physicsnemo.sym.utils.io import ValidatorPlotter
from physicsnemo.sym.key import Key
from physicsnemo.sym.node import Node

import matplotlib.pyplot as plt
plt.rcParams['image.cmap'] = 'jet'
```

### 2. Define the Main Function and Physics
- Use the `@physicsnemo.sym.main` decorator for configuration.
- Instantiate the 2D steady state Navier-Stokes equations.
- Instantiate neural network architecture (fully-connected)
- Create PhysicsNeMo nodes

```python
@physicsnemo.sym.main(config_path="conf", config_name="config_chip_2d")
def run(cfg: PhysicsNeMoConfig) -> None:
    ns = NavierStokes( FIXME ) # Define the Navier-Stokes equations
    normal_dot_vel = NormalDotVec(["u", "v"])
    flow_net = instantiate_arch(
        input_keys=[Key("x"), Key("y")],
        output_keys=[Key("u"), Key("v"), Key("p")],
        cfg=cfg.arch.fully_connected,
    )
    nodes = (
        ns.make_nodes()
        + normal_dot_vel.make_nodes()
        + [flow_net.make_node(name="flow_network")]
    )
```

### 3. Define Geometry and Domain
- Define the geometry parameters and ranges 
- Define the inlet, outlet, channel, and block 
- Create the geometry by subtracting block from the channel 
    - Note that channel and rectangle geometry objects are different. When sampling from the boundaries, no points will be samples from the two ends of a channel.
- Define the parameterized integral line
    - This will be used to enforce conservation of total flow through random Integral continuity planes 

```python
    channel_length = (-2.5, 2.5)
    channel_width = (-0.5, 0.5)
    chip_pos = -1.0
    chip_height = 0.6
    chip_width = 1.0
    inlet_vel = 1.5

    x, y = Symbol("x"), Symbol("y")
    channel = Channel2D((channel_length[0], channel_width[0]), (channel_length[1], channel_width[1]))
    rec = Rectangle((chip_pos, channel_width[0]), (chip_pos + chip_width, channel_width[0] + chip_height))
    geo = channel - rec
    
    inlet = Line((channel_length[0], channel_width[0]), (channel_length[0], channel_width[1]), normal=1)
    outlet = Line((channel_length[1], channel_width[0]), (channel_length[1], channel_width[1]), normal=1)
    x_pos = Symbol("x_pos")
    integral_line = Line((x_pos, channel_width[0]), (x_pos, channel_width[1]), 1)
    x_pos_range = {x_pos: lambda batch_size: np.full((batch_size, 1), np.random.uniform(channel_length[0], channel_length[1]))}

    domain = Domain()
```

### 4. Add Constraints
- Define the inlet constraint
    - Parabolic u-velocity, zero v-velocity
- Define the outlet constraint 
    - Zero pressure
- Define the wall constraint
    - No-slip walls: zero u-velocity and v-velocity
#### Inlet (Parabolic Profile)
```python
    inlet_parabola = parabola(y, channel_width[0], channel_width[1], inlet_vel)
    inlet = PointwiseBoundaryConstraint(
        nodes=nodes,
        geometry=inlet,
        outvar={ FIXME }, # Fill in: Add inlet boundary condition here
        batch_size=cfg.batch_size.inlet,
    )
    domain.add_constraint(inlet, "inlet")
```

#### Outlet (Pressure)
```python
    outlet = PointwiseBoundaryConstraint(
        nodes=nodes,
        geometry=outlet,
        outvar={ FIXME }, # Fill in: Add outlet boundary condition here
        batch_size=cfg.batch_size.outlet,
        criteria=Eq(x, channel_length[1]),
    )
    domain.add_constraint(outlet, "outlet")
```

#### No-Slip (Walls and Chip)
```python
    no_slip = PointwiseBoundaryConstraint(
        nodes=nodes,
        geometry=geo,
        outvar={ FIXME }, # Fill in: Add no slip boundary condition here
        batch_size=cfg.batch_size.no_slip,
    )
    domain.add_constraint(no_slip, "no_slip")
```

#### Interior Constraints (Physics)
- Define the interior constraint
    - Conservation of continuity and momentum 
- Use the signed distance function (SDF) of the interior to mitigate the difficulty of learning sharp local gradients 
    - This will weight losses lower on sharp gradients or discontinuous areas of the domain
- Signed distance function is defined as
$$ \begin{align*} S(x) =  \begin{cases} d(x,\partial D), \quad x \in D \\ -d(x,\partial D), \quad x \in D^c  \end{cases}\end{align*}$$
where d is the distance function, and D is domain.
The below figure shows a batch of interior point cloud , color coded with SDF . 
SDF is used for weighting the continuity and momentum equations for better convergence
<center><img src="images/chip_2d_sdf.png" alt="Drawing" style="width:500px" /></center

```python
    interior = PointwiseInteriorConstraint(
        nodes=nodes,
        geometry=geo,
        outvar={ FIXME }, # Fill in: Add PDE constraint here
        batch_size=cfg.batch_size.interior,
        lambda_weighting={
            "continuity": 2 * Symbol("sdf"),
            "momentum_x": 2 * Symbol("sdf"),
            "momentum_y": 2 * Symbol("sdf"),
        },
    )
    domain.add_constraint(interior, "interior")
```

#### Integral Continuity Constraint
- In addition to solving the NS equations in differential form, specifying the volumetric flow rate through some integral continuity lines or planes that are located across the channel significantly speeds up the convergence rate and improves accuracy.
- Define a callable for sampling criteria
- Define the integral constraint 
    - At each training iteration, this will create multiple random lines throughout the channel for which the total flow is computed, and adds a term to the loss function to enforce the total flow passing through each of those lines to be equal to the total inflow 
    - Integral_batch_size: Number of points within each line, to be used for Monte Carlo approximation of total flow 
    - batch_size: Number of random lines at each iteration 

```python
    def integral_criteria(invar, params):
        sdf = geo.sdf(invar, params)
        return np.greater(sdf["sdf"], 0)

    integral_continuity = IntegralBoundaryConstraint(
        nodes=nodes,
        geometry=integral_line,
        outvar={"normal_dot_vel": 1},
        batch_size=cfg.batch_size.num_integral_continuity,
        integral_batch_size=cfg.batch_size.integral_continuity,
        lambda_weighting={"normal_dot_vel": 1},
        criteria=integral_criteria,
        parameterization=x_pos_range,
    )
    domain.add_constraint(integral_continuity, "integral_continuity")
```

### 5. Add Validation
- Import and process the validation data
    - Specify a proper mapping 
    - Perform the required transformations to the data to match units, scales 
    - Define a dictionary of input and output variables for the validator
- Define the validator

```python
    file_path = "examples_sym/examples/chip_2d/openfoam/2D_chip_fluid0.csv"
    if os.path.exists(to_absolute_path(file_path)):
        mapping = {"Points:0": "x", "Points:1": "y", "U:0": "u", "U:1": "v", "p": "p"}
        openfoam_var = csv_to_dict(to_absolute_path(file_path), mapping)
        openfoam_var["x"] -= 2.5  # normalize pos
        openfoam_var["y"] -= 0.5
        openfoam_invar_numpy = {key: value for key, value in openfoam_var.items() if key in ["x", "y"]}
        openfoam_outvar_numpy = {key: value for key, value in openfoam_var.items() if key in ["u", "v", "p"]}
        openfoam_validator = PointwiseValidator(
            nodes=nodes,
            invar=openfoam_invar_numpy,
            true_outvar=openfoam_outvar_numpy,
            plotter=ValidatorPlotter(),
        )
        domain.add_validator(openfoam_validator)
    else:
        warnings.warn(
            f"Directory {file_path} does not exist. Will skip adding validators. Please download the additional files from NGC https://catalog.ngc.nvidia.com/orgs/nvidia/teams/physicsnemo/resources/physicsnemo_sym_examples_supplemental_materials"
        )
```

### 6. Create and Run the Solver
```python
    slv = Solver(cfg, domain)
    slv.solve()
```

1. Fill in all missing parts
2. Place the configuration file in the conf directory
3. Run the script:

In [1]:
!python chip_2d_l1.py

/usr/local/lib/python3.12/dist-packages/hydra/_internal/hydra.py:119: UserWarning: Future Hydra versions will no longer change working directory at job runtime by default.
See https://hydra.cc/docs/1.2/upgrades/1.1_to_1.2/changes_to_job_working_dir/ for more information.
  ret = run_job(
[W311 06:10:39.110470016 init.cpp:779] Warning: nvfuser is no longer supported in torch script, use _jit_set_nvfuser_single_node_mode is deprecated and a no-op (function operator())
[W311 06:10:39.110505394 init.cpp:767] Warning: nvfuser is no longer supported in torch script, use _jit_set_nvfuser_enabled is deprecated and a no-op (function operator())
[06:10:39] - JIT using the NVFuser TorchScript backend
[06:10:39] - JitManager: {'_enabled': True, '_arch_mode': <JitArchMode.ONLY_ACTIVATION: 1>, '_use_nvfuser': True, '_autograd_nodes': False}
[06:10:39] - GraphManager: {'_func_arch': False, '_debug': False, '_func_arch_allow_partial_hessian': True}
[06:10:39] - AmpManager: {'_enabled': False, '_mode':

## Level 2 Multiple Blocks in Channel

In Level 2, we extend the problem to include **multiple rectangular blocks** in the channel. This creates a more complex flow pattern with:
- **Multiple wake regions** behind each block
- **Flow interference** between blocks
- **Recirculation zones** in the gaps
- **More challenging geometry** for the neural network to learn

This is more realistic for applications like heat exchangers with multiple fins, urban wind flow around buildings, or flow through porous media.

### Mathematical Formulation

We use the same 2D steady-state Navier-Stokes equations:

$$\begin{align*} \text{continuity: }\quad\frac{\partial u}{\partial x} + \frac{\partial v}{\partial y}  = 0\\\
\text{momentum-x: }\quad u\frac{\partial u}{\partial x} + v\frac{\partial u}{\partial y} +\frac{1}{\rho}\frac{\partial p}{\partial x} - \nu \frac{\partial^2 u}{\partial x^2} - \nu \frac{\partial^2 u}{\partial y^2} = 0 \\\
\text{momentum-y: } \quad u\frac{\partial v}{\partial x} + v\frac{\partial v}{\partial y} +\frac{1}{\rho}\frac{\partial p}{\partial y} - \nu \frac{\partial^2 v}{\partial x^2} - \nu \frac{\partial^2 v}{\partial y^2} = 0 \\\
\end{align*}
$$

But now the domain excludes **three rectangular blocks** instead of one.

### Problem Setup

**Domain**: Channel with dimensions similar to Level 1

**Three Blocks Configuration**:
- **Block 1** (upstream): position (-1.0, bottom), size (0.6 × 0.4)
- **Block 2** (middle): position (0.2, bottom), size (0.5 × 0.5)
- **Block 3** (downstream): position (1.2, bottom), size (0.4 × 0.35)

**Boundary Conditions** (same as Level 1):
- **Inlet**: Parabolic velocity profile
- **Outlet**: Zero pressure
- **Walls and blocks**: No-slip (u = v = 0)

**Fluid Properties**:
- Density: $\rho = 1.0$
- Viscosity: $\nu = 0.02$
- Inlet velocity: $u_{max} = 1.5$

### Implementation Steps for Level 2

#### 1. Import Libraries (same as Level 1)

```python
import os
import warnings

import numpy as np
from sympy import Symbol, Eq

import physicsnemo.sym
from physicsnemo.sym.hydra import to_absolute_path, instantiate_arch
from physicsnemo.sym.hydra.config import PhysicsNeMoConfig
from physicsnemo.sym.solver import Solver
from physicsnemo.sym.domain import Domain
from physicsnemo.sym.geometry.primitives_2d import Rectangle, Line, Channel2D
from physicsnemo.sym.utils.sympy.functions import parabola
from physicsnemo.sym.eq.pdes.navier_stokes import NavierStokes
from physicsnemo.sym.eq.pdes.basic import NormalDotVec
from physicsnemo.sym.domain.constraint import (
    PointwiseBoundaryConstraint,
    PointwiseInteriorConstraint,
    IntegralBoundaryConstraint,
)
from physicsnemo.sym.key import Key
```

#### 2. Define Main Function and Create Multiple Blocks

```python
@physicsnemo.sym.main(config_path="conf", config_name="config_chip_2d")
def run(cfg: PhysicsNeMoConfig) -> None:
    # Define Navier-Stokes equations
    ns = NavierStokes( FIXME ) # Fill in: same as Level 1
    normal_dot_vel = NormalDotVec(["u", "v"])
    
    # Create neural network
    flow_net = instantiate_arch(
        input_keys=[Key("x"), Key("y")],
        output_keys=[Key("u"), Key("v"), Key("p")],
        cfg=cfg.arch.fully_connected,
    )
    
    nodes = (
        ns.make_nodes()
        + normal_dot_vel.make_nodes()
        + [flow_net.make_node(name="flow_network")]
    )
    
    # Define geometry parameters
    channel_length = (-2.5, 2.5)
    channel_width = (-0.5, 0.5)
    inlet_vel = 1.5
    
    x, y = Symbol("x"), Symbol("y")
    
    # Create channel
    channel = Channel2D(
        (channel_length[0], channel_width[0]), 
        (channel_length[1], channel_width[1])
    )
    
    # Create three rectangular blocks
    # Block 1: upstream, larger block
    block1 = Rectangle(
        FIXME # Fill in: (x_start, y_start), (x_end, y_end)
    )
    
    # Block 2: middle block, tallest
    block2 = Rectangle(
        FIXME # Fill in: (x_start, y_start), (x_end, y_end)
    )
    
    # Block 3: downstream, smallest block
    block3 = Rectangle(
        FIXME # Fill in: (x_start, y_start), (x_end, y_end)
    )
    
    # Create final geometry by subtracting all blocks from channel
    geo = FIXME # Fill in: channel minus all three blocks
```

#### 3. Define Inlet and Outlet

```python
    # Define inlet and outlet (same as Level 1)
    inlet = Line(
        (channel_length[0], channel_width[0]), 
        (channel_length[0], channel_width[1]), 
        normal=1
    )
    outlet = Line(
        (channel_length[1], channel_width[0]), 
        (channel_length[1], channel_width[1]), 
        normal=1
    )
    
    # Integral line for continuity
    x_pos = Symbol("x_pos")
    integral_line = Line(
        (x_pos, channel_width[0]), 
        (x_pos, channel_width[1]), 
        1
    )
    x_pos_range = {
        x_pos: lambda batch_size: np.full(
            (batch_size, 1), 
            np.random.uniform(channel_length[0], channel_length[1])
        )
    }
    
    domain = Domain()
```

#### 4. Add Constraints (similar to Level 1)

```python
    # Inlet constraint: parabolic velocity profile
    inlet_parabola = parabola(y, channel_width[0], channel_width[1], inlet_vel)
    inlet_constraint = PointwiseBoundaryConstraint(
        nodes=nodes,
        geometry=inlet,
        outvar={ FIXME }, # Fill in: same as Level 1
        batch_size=cfg.batch_size.inlet,
    )
    domain.add_constraint(inlet_constraint, "inlet")
    
    # Outlet constraint: zero pressure
    outlet_constraint = PointwiseBoundaryConstraint(
        nodes=nodes,
        geometry=outlet,
        outvar={ FIXME }, # Fill in: same as Level 1
        batch_size=cfg.batch_size.outlet,
        criteria=Eq(x, channel_length[1]),
    )
    domain.add_constraint(outlet_constraint, "outlet")
    
    # No-slip constraint: walls and all three blocks
    no_slip = PointwiseBoundaryConstraint(
        nodes=nodes,
        geometry=geo,
        outvar={ FIXME }, # Fill in: same as Level 1
        batch_size=cfg.batch_size.no_slip,
    )
    domain.add_constraint(no_slip, "no_slip")
    
    # Interior constraint with SDF weighting
    interior = PointwiseInteriorConstraint(
        nodes=nodes,
        geometry=geo,
        outvar={ FIXME }, # Fill in: same as Level 1
        batch_size=cfg.batch_size.interior,
        lambda_weighting={
            "continuity": 2 * Symbol("sdf"),
            "momentum_x": 2 * Symbol("sdf"),
            "momentum_y": 2 * Symbol("sdf"),
        },
    )
    domain.add_constraint(interior, "interior")
    
    # Integral continuity constraint
    def integral_criteria(invar, params):
        sdf = geo.sdf(invar, params)
        return np.greater(sdf["sdf"], 0)
    
    integral_continuity = IntegralBoundaryConstraint(
        nodes=nodes,
        geometry=integral_line,
        outvar={"normal_dot_vel": 1},
        batch_size=cfg.batch_size.num_integral_continuity,
        integral_batch_size=cfg.batch_size.integral_continuity,
        lambda_weighting={"normal_dot_vel": 1},
        criteria=integral_criteria,
        parameterization=x_pos_range,
    )
    domain.add_constraint(integral_continuity, "integral_continuity")
    
    # Create and run solver
    slv = Solver(cfg, domain)
    slv.solve()
```

1. Fill in all missing parts
2. Place the configuration file in the conf directory
3. Run the script:

In [2]:
!python chip_2d_l2.py

/usr/local/lib/python3.12/dist-packages/hydra/_internal/hydra.py:119: UserWarning: Future Hydra versions will no longer change working directory at job runtime by default.
See https://hydra.cc/docs/1.2/upgrades/1.1_to_1.2/changes_to_job_working_dir/ for more information.
  ret = run_job(
[W311 06:30:05.474178925 init.cpp:779] Warning: nvfuser is no longer supported in torch script, use _jit_set_nvfuser_single_node_mode is deprecated and a no-op (function operator())
[W311 06:30:05.474225742 init.cpp:767] Warning: nvfuser is no longer supported in torch script, use _jit_set_nvfuser_enabled is deprecated and a no-op (function operator())
[06:30:05] - JIT using the NVFuser TorchScript backend
[06:30:05] - JitManager: {'_enabled': True, '_arch_mode': <JitArchMode.ONLY_ACTIVATION: 1>, '_use_nvfuser': True, '_autograd_nodes': False}
[06:30:05] - GraphManager: {'_func_arch': False, '_debug': False, '_func_arch_allow_partial_hessian': True}
[06:30:05] - AmpManager: {'_enabled': False, '_mode':

## Level 3 Time-Dependent Flow

In Level 3, we transition from **steady-state** to **time-dependent** (unsteady) Navier-Stokes equations. This introduces:
- **Temporal evolution** of the flow field
- **Vortex shedding** (Karman vortex street) at higher Reynolds numbers
- **Periodic oscillations** in the wake
- **Initial conditions** in addition to boundary conditions
- **More complex physics** requiring time-domain training

This is crucial for applications involving transient flows, oscillating structures, pulsatile flows, and dynamic aerodynamics.

### Mathematical Formulation

The 2D **unsteady** (time-dependent) Navier-Stokes equations:

$$\begin{align*} 
\text{continuity: }\quad\frac{\partial u}{\partial x} + \frac{\partial v}{\partial y}  = 0\\
\text{momentum-x: }\quad \frac{\partial u}{\partial t} + u\frac{\partial u}{\partial x} + v\frac{\partial u}{\partial y} +\frac{1}{\rho}\frac{\partial p}{\partial x} - \nu \left(\frac{\partial^2 u}{\partial x^2} + \frac{\partial^2 u}{\partial y^2}\right) = 0 \\
\text{momentum-y: } \quad \frac{\partial v}{\partial t} + u\frac{\partial v}{\partial x} + v\frac{\partial v}{\partial y} +\frac{1}{\rho}\frac{\partial p}{\partial y} - \nu \left(\frac{\partial^2 v}{\partial x^2} + \frac{\partial^2 v}{\partial y^2}\right) = 0 \\
\end{align*}
$$

**Key difference**: Addition of time derivatives $\frac{\partial u}{\partial t}$ and $\frac{\partial v}{\partial t}$.

### Problem Setup

**Domain**: Same channel with single block (like Level 1)

**Block Configuration**:
- Single rectangular block at position (-1.0, bottom), size (1.0 × 0.6)

**Time Range**: $t \in [0, 10]$ seconds

**Increased Reynolds Number**:
- To observe vortex shedding, we increase the inlet velocity or decrease viscosity
- Reynolds number: $Re = \frac{U L}{\nu} \approx 100$ (transitional regime)
- Fluid viscosity: $\nu = 0.01$ (reduced from 0.02)
- Inlet velocity: $u_{max} = 1.5$

**Initial Conditions** (t = 0):
- $u(x, y, 0) = 0$ (fluid at rest)
- $v(x, y, 0) = 0$
- $p(x, y, 0) = 0$

**Boundary Conditions** (all times):
- **Inlet**: Parabolic velocity profile (constant in time)
- **Outlet**: Zero pressure
- **Walls and block**: No-slip (u = v = 0)

### Expected Physics: Vortex Shedding

At moderate Reynolds numbers (~40-200), flow past a bluff body exhibits **vortex shedding**:
1. **Startup**: Flow accelerates from rest
2. **Vortex formation**: Vortices form behind the block
3. **Periodic shedding**: Vortices alternately detach from top and bottom (Karman vortex street)
4. **Oscillating wake**: Periodic velocity and pressure fluctuations
5. **Strouhal number**: Characteristic frequency $St = \frac{fL}{U} \approx 0.2$ for cylinders

### Implementation Steps for Level 3

#### 1. Import Libraries (add time-related imports)

```python
import os
import warnings

import numpy as np
from sympy import Symbol, Eq

import physicsnemo.sym
from physicsnemo.sym.hydra import to_absolute_path, instantiate_arch
from physicsnemo.sym.hydra.config import PhysicsNeMoConfig
from physicsnemo.sym.solver import Solver
from physicsnemo.sym.domain import Domain
from physicsnemo.sym.geometry.primitives_2d import Rectangle, Line, Channel2D
from physicsnemo.sym.utils.sympy.functions import parabola
from physicsnemo.sym.eq.pdes.navier_stokes import NavierStokes
from physicsnemo.sym.eq.pdes.basic import NormalDotVec
from physicsnemo.sym.domain.constraint import (
    PointwiseBoundaryConstraint,
    PointwiseInteriorConstraint,
    IntegralBoundaryConstraint,
)
from physicsnemo.sym.key import Key
```

#### 2. Define Time-Dependent Navier-Stokes

```python
@physicsnemo.sym.main(config_path="conf", config_name="config_chip_2d")
def run(cfg: PhysicsNeMoConfig) -> None:
    # Define UNSTEADY Navier-Stokes equations (time=True)
    ns = NavierStokes( FIXME ) # Fill in : Correct variables for UNSTEADY Navier-Stokes equations
    normal_dot_vel = NormalDotVec(["u", "v"])
    
    # Create neural network with TIME as input
    flow_net = instantiate_arch(
        input_keys=[FIXME, FIXME, FIXME],  # Fill in: Input for UNSTEADY Navier-Stokes equations
        output_keys=[Key("u"), Key("v"), Key("p")],
        cfg=cfg.arch.fully_connected,
    )
    
    nodes = (
        ns.make_nodes()
        + normal_dot_vel.make_nodes()
        + [flow_net.make_node(name="flow_network")]
    )
    
    # Define geometry parameters
    channel_length = (-2.5, 2.5)
    channel_width = (-0.5, 0.5)
    chip_pos = -1.0
    chip_height = 0.6
    chip_width = 1.0
    inlet_vel = 1.5
    
    x, y, t = Symbol("x"), Symbol("y"), Symbol("t")
    
    # Create geometry (same as Level 1)
    channel = Channel2D(
        (channel_length[0], channel_width[0]), 
        (channel_length[1], channel_width[1])
    )
    rec = Rectangle(
        (chip_pos, channel_width[0]), 
        (chip_pos + chip_width, channel_width[0] + chip_height)
    )
    geo = channel - rec
```

#### 3. Define Time Range and Initial Conditions

```python
    # Define time range
    time_range = {t: (0, 10)}  # Simulate from t=0 to t=10 seconds
    
    # Define inlet and outlet
    inlet = Line(
        (channel_length[0], channel_width[0]), 
        (channel_length[0], channel_width[1]), 
        normal=1
    )
    outlet = Line(
        (channel_length[1], channel_width[0]), 
        (channel_length[1], channel_width[1]), 
        normal=1
    )
    
    # Integral line (time-dependent)
    x_pos = Symbol("x_pos")
    integral_line = Line(
        (x_pos, channel_width[0]), 
        (x_pos, channel_width[1]), 
        1
    )
    x_pos_range = {
        x_pos: lambda batch_size: np.full(
            (batch_size, 1), 
            np.random.uniform(channel_length[0], channel_length[1])
        )
    }
    
    domain = Domain()
```

#### 4. Add Initial Condition (NEW for time-dependent)

```python
    # Initial condition: fluid at rest at t=0
    IC = PointwiseInteriorConstraint(
        nodes=nodes,
        geometry=geo,
        outvar={ FIXME }, # Fill in: Add Initial Condition here
        batch_size=cfg.batch_size.IC,
        parameterization={t: 0},  # Fix time at t=0
    )
    domain.add_constraint(IC, "IC")
```

#### 5. Add Boundary Constraints (time-dependent)

```python
    # Inlet constraint: parabolic velocity (for all times)
    inlet_parabola = parabola(y, channel_width[0], channel_width[1], inlet_vel)
    inlet_constraint = PointwiseBoundaryConstraint(
        nodes=nodes,
        geometry=inlet,
        outvar={ FIXME }, # Fill in: same as Level 1 
        batch_size=cfg.batch_size.inlet,
        parameterization=time_range,  # Sample over time
    )
    domain.add_constraint(inlet_constraint, "inlet")
    
    # Outlet constraint: zero pressure (for all times)
    outlet_constraint = PointwiseBoundaryConstraint(
        nodes=nodes,
        geometry=outlet,
        outvar={ FIXME }, # Fill in: same as Level 1 
        batch_size=cfg.batch_size.outlet,
        criteria=Eq(x, channel_length[1]),
        parameterization=time_range,
    )
    domain.add_constraint(outlet_constraint, "outlet")
    
    # No-slip constraint: walls and block (for all times)
    no_slip = PointwiseBoundaryConstraint(
        nodes=nodes,
        geometry=geo,
        outvar={ FIXME }, # Fill in: same as Level 1 
        batch_size=cfg.batch_size.no_slip,
        parameterization=time_range,
    )
    domain.add_constraint(no_slip, "no_slip")
```

#### 6. Add Interior Constraint (time-dependent PDE)

```python
    # Interior constraint: unsteady Navier-Stokes equations
    interior = PointwiseInteriorConstraint(
        nodes=nodes,
        geometry=geo,
        outvar={}, # Fill in: same as Level 1 
        batch_size=cfg.batch_size.interior,
        lambda_weighting={
            "continuity": 2 * Symbol("sdf"),
            "momentum_x": 2 * Symbol("sdf"),
            "momentum_y": 2 * Symbol("sdf"),
        },
        parameterization=time_range,  # Sample over space AND time
    )
    domain.add_constraint(interior, "interior")
    
    # Integral continuity constraint (time-dependent)
    def integral_criteria(invar, params):
        sdf = geo.sdf(invar, params)
        return np.greater(sdf["sdf"], 0)
    
    integral_continuity = IntegralBoundaryConstraint(
        nodes=nodes,
        geometry=integral_line,
        outvar={"normal_dot_vel": 1},
        batch_size=cfg.batch_size.num_integral_continuity,
        integral_batch_size=cfg.batch_size.integral_continuity,
        lambda_weighting={"normal_dot_vel": 1},
        criteria=integral_criteria,
        parameterization={**x_pos_range, **time_range},  # Both space and time
    )
    domain.add_constraint(integral_continuity, "integral_continuity")
    
    # Create and run solver
    slv = Solver(cfg, domain)
    slv.solve()
```

1. Fill in all missing parts
2. Place the configuration file in the conf directory
3. Run the script:

In [3]:
!python chip_2d_l3.py

/usr/local/lib/python3.12/dist-packages/hydra/_internal/hydra.py:119: UserWarning: Future Hydra versions will no longer change working directory at job runtime by default.
See https://hydra.cc/docs/1.2/upgrades/1.1_to_1.2/changes_to_job_working_dir/ for more information.
  ret = run_job(
[W311 06:39:56.824113023 init.cpp:779] Warning: nvfuser is no longer supported in torch script, use _jit_set_nvfuser_single_node_mode is deprecated and a no-op (function operator())
[W311 06:39:56.824165198 init.cpp:767] Warning: nvfuser is no longer supported in torch script, use _jit_set_nvfuser_enabled is deprecated and a no-op (function operator())
[06:39:56] - JIT using the NVFuser TorchScript backend
[06:39:56] - JitManager: {'_enabled': True, '_arch_mode': <JitArchMode.ONLY_ACTIVATION: 1>, '_use_nvfuser': True, '_autograd_nodes': False}
[06:39:56] - GraphManager: {'_func_arch': False, '_debug': False, '_func_arch_allow_partial_hessian': True}
[06:39:56] - AmpManager: {'_enabled': False, '_mode':

In [1]:
!python chip_2d_l3.py hydra.run.dir="outputs/chip_2d_l3_grad_norm"

/usr/local/lib/python3.12/dist-packages/hydra/_internal/hydra.py:119: UserWarning: Future Hydra versions will no longer change working directory at job runtime by default.
See https://hydra.cc/docs/1.2/upgrades/1.1_to_1.2/changes_to_job_working_dir/ for more information.
  ret = run_job(
[W311 07:39:23.043906483 init.cpp:779] Warning: nvfuser is no longer supported in torch script, use _jit_set_nvfuser_single_node_mode is deprecated and a no-op (function operator())
[W311 07:39:23.043946504 init.cpp:767] Warning: nvfuser is no longer supported in torch script, use _jit_set_nvfuser_enabled is deprecated and a no-op (function operator())
[07:39:23] - JIT using the NVFuser TorchScript backend
[07:39:23] - JitManager: {'_enabled': True, '_arch_mode': <JitArchMode.ONLY_ACTIVATION: 1>, '_use_nvfuser': True, '_autograd_nodes': False}
[07:39:23] - GraphManager: {'_func_arch': False, '_debug': False, '_func_arch_allow_partial_hessian': True}
[07:39:23] - AmpManager: {'_enabled': False, '_mode':

--- 

Don't forget to check out additional [Open Hackathons Resources](https://www.openhackathons.org/s/technical-resources) and join our [OpenACC and Hackathons Slack Channel](https://www.openacc.org/community#slack) to share your experience and get more help from the community.

---

# Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials may include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.